# 00 — Offline execution check: FICTIONAL data only

This notebook checks the entire software path on generated arrays. It requires no Earth Engine login and produces **no Bangladesh scientific result**. CPU is sufficient. Real imagery, quality, geographic boundaries and GPU execution are not validated by this fixture.

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util, json
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent, Path("/content/cht-soil-moisture-gapfilling")]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None and IN_COLAB:
    ROOT = Path("/content/cht-soil-moisture-gapfilling")
    subprocess.run(["git", "clone", "--branch", "research-notebooks-v2", "--single-branch",
                    "https://github.com/Bipul-roy-AI/cht-soil-moisture-gapfilling.git", str(ROOT)], check=True)
if ROOT is None:
    raise RuntimeError("Open this notebook from the cloned repository, or run it in Colab.")
ROOT = ROOT.resolve()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[gee,notebooks]"], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)


Project: /workspace/cht-soil-moisture-gapfilling


In [2]:
import tempfile
from chtsm.synthetic import make_fixture
from chtsm.models import fit_experiment, freeze_experiment, evaluate_final
from chtsm.gaps import run_gap_experiments
frame, config = make_fixture(ROOT / "configs/study_v2.json")
if importlib.util.find_spec("xgboost") is None:
    config["algorithms"] = ["random_forest"]
RUN = Path(tempfile.mkdtemp(prefix="cht_fictional_check_")) / "run"
metadata, validation_table = fit_experiment(frame, config, RUN, software_check=True)
print("FICTIONAL SOFTWARE CHECK ONLY")
print("Rows:", len(frame), "| validation-selected model:", metadata["selected_model"])
display(validation_table[["name", "validation_macro_cell_rmse"]])

Fitting random_forest / climate / seed 42; selection uses validation only


Fitting random_forest / climate_raw_sar / seed 42; selection uses validation only


Fitting random_forest / climate_rtc_sar / seed 42; selection uses validation only


Fitting xgboost / climate / seed 42; selection uses validation only


Fitting xgboost / climate_raw_sar / seed 42; selection uses validation only


Fitting xgboost / climate_rtc_sar / seed 42; selection uses validation only


FICTIONAL SOFTWARE CHECK ONLY
Rows: 8764 | validation-selected model: random_forest_climate_seed42


,name,validation_macro_cell_rmse
0,global_mean,0.047922
1,seasonal_cell,0.019245
2,era5_linear,0.029912
3,random_forest_climate_seed42,0.011688
4,random_forest_climate_raw_sar_seed42,0.011788
5,random_forest_climate_rtc_sar_seed42,0.011781
6,xgboost_climate_seed42,0.015852
7,xgboost_climate_raw_sar_seed42,0.015802
8,xgboost_climate_rtc_sar_seed42,0.015796


In [3]:
freeze_experiment(RUN, "Fictional software verification only; all prespecified methods; primary outcome macro-cell RMSE; no field or satellite observations.")
metrics, reconstruction = evaluate_final(frame, RUN, allow_final_test=True)
gaps = run_gap_experiments(frame, RUN, lengths=(1, 3), seeds=(42,))
assert reconstruction.loc[reconstruction.target_sm.notna(), "complete_sm"].equals(
    reconstruction.loc[reconstruction.target_sm.notna(), "target_sm"])
print("Training, calibration, freeze, evaluation, reconstruction and gap tests completed.")
print("Artifacts:", RUN)
print("No accuracy result from this notebook may be described as Bangladesh performance.")

Training, calibration, freeze, evaluation, reconstruction and gap tests completed.
Artifacts: /tmp/cht_fictional_check_lgl0j97w/run
No accuracy result from this notebook may be described as Bangladesh performance.
